# Olist E-Commerce — Data Cleaning

Cleans and type-casts the raw Olist CSV exports, then optionally loads the
cleaned tables into PostgreSQL for the dimensional model in `sql/`.

**Requirements:** `pandas`, and `sqlalchemy` + `psycopg2-binary` if `LOAD_TO_DB`
is enabled below. See `requirements.txt`.

**Before running:** place the raw Olist CSVs in a `files/` folder next to this
notebook.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("files")

# Set to True to write the cleaned tables into PostgreSQL at the end of this
# notebook. Credentials are read from environment variables (or prompted for
# the password), so nothing sensitive is hardcoded in the notebook itself.
LOAD_TO_DB = False

In [ ]:
def load_csv(filename: str, **kwargs) -> pd.DataFrame:
    """Read a CSV from DATA_DIR, forwarding kwargs to pd.read_csv."""
    return pd.read_csv(DATA_DIR / filename, **kwargs)


def load_to_postgres(df: pd.DataFrame, table_name: str, engine) -> None:
    """Write a cleaned DataFrame to PostgreSQL, replacing the table if it exists."""
    df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"Loaded {len(df):,} rows into '{table_name}'")

## Customers

In [ ]:
df_customers = load_csv(
    "olist_customers_dataset.csv",
    dtype={"customer_zip_code_prefix": "str"},
)
df_customers.info()
df_customers.head()

## Geolocation

In [ ]:
df_geolocation = load_csv(
    "olist_geolocation_dataset.csv",
    dtype={"geolocation_zip_code_prefix": "str"},
)
df_geolocation.info()
df_geolocation.head()

## Order Items

In [ ]:
df_order_items = load_csv("olist_order_items_dataset.csv")
df_order_items.info()
df_order_items.head()

In [ ]:
# shipping_limit_date arrives as a string; cast it to a real datetime
df_order_items["shipping_limit_date"] = pd.to_datetime(df_order_items["shipping_limit_date"])
df_order_items.dtypes

## Order Payments

In [ ]:
df_order_payments = load_csv("olist_order_payments_dataset.csv")
df_order_payments.info()
df_order_payments.head()

In [ ]:
# Bucket payment_installments into readable groups in a single pass.
# Using -1 as the lower bound lets 0 installments fall cleanly into its own
# "No Installment" bin, instead of needing a separate assignment + pd.cut step.
installment_bins = [-1, 0, 2, 6, 12, 24]
installment_labels = [
    "No Installment",
    "Short (1-2)",
    "Medium (3-6)",
    "Long (7-12)",
    "Very Long (13-24)",
]

df_order_payments["installment_group"] = pd.cut(
    df_order_payments["payment_installments"],
    bins=installment_bins,
    labels=installment_labels,
)

df_order_payments.head()

## Order Reviews

In [ ]:
df_order_review = load_csv("olist_order_reviews_dataset.csv")
df_order_review.info()
df_order_review.head()

In [ ]:
df_order_review.isna().sum()

In [ ]:
df_order_review["review_comment_title"] = df_order_review["review_comment_title"].fillna("No Title")
df_order_review["review_comment_message"] = df_order_review["review_comment_message"].fillna("No comment")

df_order_review.isna().sum()

## Orders

In [ ]:
df_order_dataset = load_csv("olist_orders_dataset.csv")
df_order_dataset.info()
df_order_dataset.head()

In [ ]:
df_order_dataset.isna().sum()

In [ ]:
cancelled_order_count = (df_order_dataset["order_status"] == "canceled").sum()
print(f"Cancelled orders: {cancelled_order_count:,}")

In [ ]:
# Explicit column names instead of a positional slice (df.columns[3:]) --
# the slice silently breaks if a column gets added, removed, or reordered
# upstream, whereas naming the columns fails loudly and obviously instead.
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
df_order_dataset[date_cols] = df_order_dataset[date_cols].apply(
    pd.to_datetime, errors="coerce"
)

df_order_dataset.dtypes

## Products

In [ ]:
df_products_dataset = load_csv("olist_products_dataset.csv")
df_products_dataset.info()
df_products_dataset.head()

In [ ]:
(df_products_dataset.isna().mean() * 100).round(2)

In [ ]:
# Products missing a category are also missing every dimension/weight field,
# i.e. they are essentially blank listings rather than a data entry slip --
df_products_dataset[df_products_dataset["product_category_name"].isna()].isna().sum()

In [ ]:
df_products_dataset["product_category_name"] = df_products_dataset["product_category_name"].fillna("Unknown")

# Placeholder length for products missing a name, matching the length of the
# literal word "unknown" used elsewhere as the categorical placeholder.
df_products_dataset["product_name_lenght"] = df_products_dataset["product_name_lenght"].fillna(len("unknown"))

df_products_dataset = df_products_dataset.fillna(
    {"product_description_lenght": 0, "product_photos_qty": 0}
)

# Median imputation for physical dimensions. Note: assigning back to the
# column (df[col] = df[col].fillna(...)) instead of df[col].fillna(...,
# inplace=True) avoids pandas' chained-assignment warning and reliably
# updates df_products_dataset in every pandas version.
dimension_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for col in dimension_cols:
    df_products_dataset[col] = df_products_dataset[col].fillna(df_products_dataset[col].median())

df_products_dataset.isna().sum()

## Sellers

In [ ]:
df_sellers_dataset = load_csv(
    "olist_sellers_dataset.csv",
    dtype={"seller_zip_code_prefix": "str"},
)
df_sellers_dataset.head()

In [ ]:
df_sellers_dataset.isna().sum()

## Product Category Translation

In [ ]:
df_product_category_translation = load_csv("product_category_name_translation.csv")
df_product_category_translation.head()

In [ ]:
df_product_category_translation.isna().sum()

In [ ]:
import os
from getpass import getpass

if LOAD_TO_DB:
    from sqlalchemy import create_engine

    db_user = os.environ.get("OLIST_DB_USER", "postgres")
    db_password = os.environ.get("OLIST_DB_PASSWORD") or getpass("Postgres password: ")
    db_host = os.environ.get("OLIST_DB_HOST", "localhost")
    db_port = os.environ.get("OLIST_DB_PORT", "5432")
    db_name = os.environ.get("OLIST_DB_NAME", "olist")

    engine = create_engine(
        f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
    )

In [ ]:
tables_to_load = {
    "olist_customers_dataset": df_customers,
    "olist_geolocation_dataset": df_geolocation,
    "olist_order_items_dataset": df_order_items,
    "olist_order_payments_dataset": df_order_payments,
    "olist_order_reviews_dataset": df_order_review,
    "olist_orders_dataset": df_order_dataset,
    "olist_products_dataset": df_products_dataset,
    "olist_sellers_dataset": df_sellers_dataset,
    "product_category_translation": df_product_category_translation,
}

if LOAD_TO_DB:
    for table_name, df in tables_to_load.items():
        load_to_postgres(df, table_name, engine)
else:
    print("LOAD_TO_DB is False — skipping database load.")